In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report, accuracy_score
import joblib
import sys
sys.path.append('..')
from app.preprocess import clean_arabic_text

In [ ]:
df = pd.read_csv('../app/data/dataset.csv')
print(f"Dataset chargé: {len(df)} exemples")


df['clean'] = df['text'].apply(clean_arabic_text)
print("\nAperçu:")
df[['text', 'clean']].head()
print(df.columns)

Dataset chargé: 4252 exemples

Aperçu:
Index(['text', 'Sentiment', 'clean'], dtype='str')


: 

In [ ]:
X = df['clean']
y = df['Sentiment']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
    #stratify=y pour maintenir la même distribution des classes dans les ensembles d'entraînement et de test
)

vectorizer = TfidfVectorizer(max_features=1000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Train: {X_train_vec.shape[0]} samples")
print(f"Test: {X_test_vec.shape[0]} samples")

Train: 3401 samples
Test: 851 samples


: 

In [ ]:
modeles = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'SVM': SVC(kernel='linear'),
    'Random Forest': RandomForestClassifier(n_estimators=100)
}

resultats = []
best_f1 = 0
best_modele = None
best_nom = None

print("="*50)
print("COMPARAISON DES MODÈLES")
print("="*50)

for nom, modele in modeles.items():
    modele.fit(X_train_vec, y_train)
    
    
    y_pred = modele.predict(X_test_vec)
    
    
    f1 = f1_score(y_test, y_pred, average='weighted')
    acc = accuracy_score(y_test, y_pred)
    
    resultats.append({
        'Modèle': nom,
        'Accuracy': round(acc, 4),
        'F1-Score': round(f1, 4)
    })
    
    print(f"\n{nom}")
    print(f"   Accuracy: {acc:.4f}")
    print(f"   F1-Score: {f1:.4f}")
    
    if f1 > best_f1:
        best_f1 = f1
        best_modele = modele
        best_nom = nom

COMPARAISON DES MODÈLES

Naive Bayes
   Accuracy: 0.8461
   F1-Score: 0.8431

Logistic Regression
   Accuracy: 0.8578
   F1-Score: 0.8563

SVM
   Accuracy: 0.8660
   F1-Score: 0.8650

Random Forest
   Accuracy: 0.8578
   F1-Score: 0.8559


: 

In [ ]:
print("\n" + "="*50)
print(" RÉSULTATS FINAUX - CLASSEMENT")
print("="*50)

resultats_df = pd.DataFrame(resultats)
resultats_df = resultats_df.sort_values('F1-Score', ascending=False)

for i, row in resultats_df.iterrows():
    print(f"{row['Modèle']:20} | F1: {row['F1-Score']:.4f} | Acc: {row['Accuracy']:.4f}")

print(f"\nMeilleur modèle: {best_nom} avec F1 = {best_f1:.4f}")


 RÉSULTATS FINAUX - CLASSEMENT
SVM                  | F1: 0.8650 | Acc: 0.8660
Logistic Regression  | F1: 0.8563 | Acc: 0.8578
Random Forest        | F1: 0.8559 | Acc: 0.8578
Naive Bayes          | F1: 0.8431 | Acc: 0.8461

Meilleur modèle: SVM avec F1 = 0.8650


: 

In [ ]:
joblib.dump(best_modele, '../app/models/best_model.pkl')
joblib.dump(vectorizer, '../app/models/vectorizer.pkl')

['../app/models/vectorizer.pkl']

: 